# Bottom-up monthly water-use experiments

This notebook reproduces the complete analysis from three source datasets stored in Google Drive. It runs the core nonnegative models, focused seasonal refinements, signed-effects sensitivity analysis, uncertainty figures, HTML reports, spatial outputs, and tests.

It does **not** use archived predictions, fitted parameters, crosswalks, or derived targets.

In [ ]:
# Mount Google Drive.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# EDIT THIS: folder containing the three required source files.
from pathlib import Path
DRIVE_INPUT_DIR = Path('/content/drive/MyDrive/CWCB/water_curve_inputs')

REQUIRED = [
    'parcels_water_2021.gpkg',
    'landcover_2021.gpkg',
    'satellite_timeseries.sqlite',
]
missing = [name for name in REQUIRED if not (DRIVE_INPUT_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f'Missing from {DRIVE_INPUT_DIR}: {missing}')
print('Found all three source datasets.')

In [ ]:
# Clone/update the analysis repository.
import subprocess
REPO_ROOT = Path('/content/cwcb-landcover-mapping')
if not REPO_ROOT.exists():
    subprocess.run(['git','clone','https://github.com/sig-gis/cwcb-landcover-mapping.git',str(REPO_ROOT)],check=True)
else:
    subprocess.run(['git','-C',str(REPO_ROOT),'pull','--ff-only'],check=True)

# The parent directory in this repository intentionally has a trailing space.
ANALYSIS_ROOT = REPO_ROOT / '00_Intial_Explorations ' / 'water_curve_bottom_up'
if not ANALYSIS_ROOT.is_dir():
    raise FileNotFoundError(ANALYSIS_ROOT)
print('Analysis root:', ANALYSIS_ROOT)

In [ ]:
# Install the scientific packages. Colab supplies GDAL/libgdal.
import sys
subprocess.run([sys.executable,'-m','pip','install','-q','numpy','scipy','scikit-learn','matplotlib','nbformat','Markdown','joblib'],check=True)
try:
    from osgeo import ogr
except ImportError:
    version=subprocess.check_output(['gdal-config','--version'],text=True).strip()
    subprocess.run([sys.executable,'-m','pip','install','-q',f'GDAL=={version}'],check=True)
print('Dependencies ready.')

In [ ]:
# Stage only the three authoritative files in the disposable Colab runtime.
import shutil
runtime_inputs = ANALYSIS_ROOT / 'inputs'
runtime_inputs.mkdir(exist_ok=True)
for name in REQUIRED:
    destination=runtime_inputs/name
    if destination.exists(): destination.unlink()
    shutil.copy2(DRIVE_INPUT_DIR/name,destination)
    print(name, destination.stat().st_size, 'bytes')

## Run all experiment threads

Each command is deterministic and limited to one numerical thread. The complete workflow generally takes a few minutes in Colab.

In [ ]:
import os
env=os.environ.copy()
env.update({'OMP_NUM_THREADS':'1','OPENBLAS_NUM_THREADS':'1','MKL_NUM_THREADS':'1','MPLCONFIGDIR':str(ANALYSIS_ROOT/'cache'/'matplotlib')})
commands=[
    [sys.executable,'-m','src.pipeline'],
    [sys.executable,'-m','src.refinement'],
    [sys.executable,'-m','src.signed_effects'],
    [sys.executable,'-m','src.landsat_only'],
    [sys.executable,'-m','src.export_model_artifacts'],
    [sys.executable,'-m','src.build_uncertainty_report'],
    [sys.executable,'-m','src.build_technical_report'],
]
for command in commands:
    print('RUNNING:', ' '.join(command))
    subprocess.run(command,cwd=ANALYSIS_ROOT,env=env,check=True)
print('All models and reports completed.')

In [ ]:
# Run integrity tests.
subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-v'],cwd=ANALYSIS_ROOT,env=env,check=True)

In [ ]:
# Display the held-out comparison tables.
import pandas as pd
from IPython.display import display
for filename in ['model_comparison.csv','refinement_model_comparison.csv','signed_model_comparison.csv']:
    print(filename)
    display(pd.read_csv(ANALYSIS_ROOT/'outputs'/filename))

In [ ]:
# Open links to the two self-contained HTML reports.
from IPython.display import FileLink, display
display(FileLink(str(ANALYSIS_ROOT/'reports'/'water_use_uncertainty_report.html')))
display(FileLink(str(ANALYSIS_ROOT/'reports'/'signed_landcover_effects_report.html')))
print('All outputs:', ANALYSIS_ROOT/'outputs')